<a href="https://colab.research.google.com/github/ZiFiID/Website/blob/main/url_blaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ URL Blaster
### Mass URL opener with no-repeat proxy rotation — runs in background
**Runtime → Run all, then fill in the form and click ▶ Start**

In [ ]:
import subprocess, sys
pkgs = ['aiohttp', 'aiofiles']
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependencies ready.')

In [ ]:
import asyncio, aiohttp, aiofiles, time, random, json, threading
from datetime import datetime
from collections import deque
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

HEADERS_POOL = [
    {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36', 'Accept-Language': 'en-US,en;q=0.9', 'Accept-Encoding': 'gzip, deflate, br', 'Connection': 'keep-alive'},
    {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 13_6) AppleWebKit/605.1.15 Version/17.0 Safari/605.1.15', 'Accept-Language': 'en-GB,en;q=0.8', 'Accept-Encoding': 'gzip, deflate, br', 'Connection': 'keep-alive'},
    {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:121.0) Gecko/20100101 Firefox/121.0', 'Accept-Language': 'en-US,en;q=0.5', 'Accept-Encoding': 'gzip, deflate, br', 'Connection': 'keep-alive'},
    {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/115.0', 'Accept-Language': 'en-US,en;q=0.7', 'Accept-Encoding': 'gzip, deflate', 'Connection': 'keep-alive'},
    {'User-Agent': 'Mozilla/5.0 (Linux; Android 13; Pixel 7) AppleWebKit/537.36 Chrome/120.0.6099.43 Mobile Safari/537.36', 'Accept-Language': 'en-US,en;q=0.9', 'Accept-Encoding': 'gzip, deflate, br', 'Connection': 'keep-alive'},
]

LOG_FILE = '/content/blaster_log.json'

stats = {'success': 0, 'failed': 0, 'total_sent': 0, 'proxy_errors': 0,
         'start_time': None, 'pause_start': 0, 'paused_total': 0}
recent_log      = deque(maxlen=20)
_stop_event     = threading.Event()
_pause_event    = threading.Event()
_blaster_thread = None
_proxy_pool_size = 0


async def fetch_free_proxies():
    sources = [
        'https://raw.githubusercontent.com/TheSpeedX/PROXY-List/master/http.txt',
        'https://raw.githubusercontent.com/clarketm/proxy-list/master/proxy-list-raw.txt',
        'https://raw.githubusercontent.com/sunny9577/proxy-scraper/master/proxies.txt',
        'https://raw.githubusercontent.com/monosans/proxy-list/main/proxies/http.txt',
        'https://raw.githubusercontent.com/ShiftyTR/Proxy-List/master/http.txt',
    ]
    proxies = set()
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=15)) as s:
        for src in sources:
            try:
                async with s.get(src) as r:
                    if r.status == 200:
                        for line in (await r.text()).strip().splitlines():
                            line = line.strip()
                            if line and ':' in line and not line.startswith('#'):
                                proxies.add(f'http://{line}')
            except Exception:
                continue
    return list(proxies)


class NoRepeatShuffler:
    def __init__(self, items):
        self._pool = list(items)
        self._bag  = []
        self._last = None

    def next(self):
        if len(self._pool) == 1:
            return self._pool[0]
        if not self._bag:
            self._bag = list(self._pool)
            random.shuffle(self._bag)
            if self._bag[0] == self._last:
                self._bag.append(self._bag.pop(0))
        pick = self._bag.pop(0)
        self._last = pick
        return pick


async def visit_url(session, url, proxy, semaphore, cfg):
    cap = cfg['concurrent']
    ctx = asyncio.nullcontext() if cap == 0 else semaphore
    async with ctx:
        while _pause_event.is_set():
            await asyncio.sleep(0.2)
        headers = random.choice(HEADERS_POOL)
        t0 = time.time()
        entry = {'ts': datetime.now().strftime('%H:%M:%S'), 'url': url, 'proxy': proxy or 'direct'}
        try:
            async with session.get(
                url, proxy=proxy, headers=headers,
                timeout=aiohttp.ClientTimeout(total=10),
                ssl=False, allow_redirects=True
            ) as resp:
                if cfg['duration'] > 0:
                    await asyncio.sleep(cfg['duration'])
                entry.update({'status': resp.status, 'time': round(time.time()-t0,2), 'ok': resp.status < 400})
                stats['success'] += 1
        except Exception as e:
            entry.update({'status': 0, 'time': round(time.time()-t0,2), 'ok': False, 'error': str(e)[:80]})
            stats['failed'] += 1
            if proxy:
                stats['proxy_errors'] += 1
        finally:
            stats['total_sent'] += 1
            recent_log.append(entry)


async def _run_loop(proxy_pool, cfg):
    shuffler  = NoRepeatShuffler(proxy_pool)
    cap       = cfg['concurrent']
    limit     = None if cap == 0 else cap
    semaphore = asyncio.Semaphore(cap if cap > 0 else 99999)
    connector = aiohttp.TCPConnector(limit=limit, ssl=False, force_close=True)
    infinite  = cfg['total'] == 0
    counter   = 0
    async with aiohttp.ClientSession(connector=connector) as session:
        pending = set()
        while not _stop_event.is_set():
            if _pause_event.is_set():
                await asyncio.sleep(0.2)
                continue
            if not infinite and counter >= cfg['total']:
                break
            url   = random.choice(cfg['urls'])
            proxy = shuffler.next()
            t = asyncio.ensure_future(visit_url(session, url, proxy, semaphore, cfg))
            pending.add(t)
            t.add_done_callback(pending.discard)
            counter += 1
            batch = cap if cap > 0 else 500
            if len(pending) >= batch:
                await asyncio.wait(pending, return_when=asyncio.FIRST_COMPLETED)
        if pending:
            await asyncio.gather(*pending, return_exceptions=True)
    async with aiofiles.open(LOG_FILE, 'w') as f:
        await f.write(json.dumps(list(recent_log), indent=2))


def _thread_target(proxy_pool, cfg):
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(_run_loop(proxy_pool, cfg))
    loop.close()


async def _get_proxy_pool(extra):
    pool = [p.strip() for p in extra if p.strip()]
    fetched = await fetch_free_proxies()
    pool.extend(fetched)
    return pool if pool else [None]


def _state():
    alive  = _blaster_thread and _blaster_thread.is_alive()
    paused = _pause_event.is_set()
    if alive and paused: return 'paused'
    if alive: return 'running'
    return 'stopped'


def render_dashboard(cfg):
    state       = _state()
    elapsed_raw = (time.time() - stats['start_time'] - stats['paused_total']) if stats['start_time'] else 0
    elapsed     = round(max(elapsed_raw, 0), 1)
    rps         = round(stats['total_sent'] / elapsed, 2) if elapsed > 0 else 0
    dot         = {'running': '🟢 RUNNING', 'paused': '🟡 PAUSED', 'stopped': '🔴 STOPPED'}[state]
    rows = ''
    for r in reversed(list(recent_log)):
        color = '#22c55e' if r.get('ok') else '#ef4444'
        rows += (
            '<tr>'
            f'<td>{r["ts"]}</td>'
            f'<td style="max-width:200px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap">{r["url"]}</td>'
            f'<td style="color:#64748b;font-size:11px;max-width:150px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap">{r["proxy"]}</td>'
            f'<td style="color:{color};font-weight:bold">{r.get("status") or "ERR"}</td>'
            f'<td>{r.get("time","")}s</td>'
            f'<td style="color:#f87171;font-size:10px">{r.get("error","")}</td>'
            '</tr>'
        )
    conc_lbl = '∞' if cfg['concurrent'] == 0 else cfg['concurrent']
    dur_lbl  = 'instant' if cfg['duration'] == 0 else f"{cfg['duration']}s"
    tot_lbl  = '∞' if cfg['total'] == 0 else cfg['total']
    return (
        '<style>'
        '.bd{font-family:"Courier New",monospace;background:#0f172a;padding:20px;border-radius:12px;color:#e2e8f0;font-size:13px}'
        '.sg{display:grid;grid-template-columns:repeat(6,1fr);gap:10px;margin-bottom:16px}'
        '.sb{background:#1e293b;border-radius:8px;padding:10px;text-align:center}'
        '.sv{font-size:24px;font-weight:bold}'
        '.sl{font-size:10px;color:#64748b;margin-top:3px}'
        'table{width:100%;border-collapse:collapse;font-size:11px}'
        'th{color:#475569;text-align:left;padding:5px 8px;border-bottom:1px solid #1e293b}'
        'td{padding:4px 8px;border-bottom:1px solid #0f172a;color:#cbd5e1}'
        '</style>'
        '<div class="bd">'
        f'<div style="font-size:17px;font-weight:bold;margin-bottom:14px;color:#38bdf8">⚡ URL Blaster &nbsp;<span style="font-size:12px">{dot}</span></div>'
        '<div class="sg">'
        f'<div class="sb"><div class="sv" style="color:#22c55e">{stats["success"]}</div><div class="sl">SUCCESS</div></div>'
        f'<div class="sb"><div class="sv" style="color:#ef4444">{stats["failed"]}</div><div class="sl">FAILED</div></div>'
        f'<div class="sb"><div class="sv" style="color:#38bdf8">{stats["total_sent"]}</div><div class="sl">SENT</div></div>'
        f'<div class="sb"><div class="sv" style="color:#f59e0b">{rps}</div><div class="sl">REQ/S</div></div>'
        f'<div class="sb"><div class="sv" style="color:#a78bfa">{elapsed}s</div><div class="sl">ELAPSED</div></div>'
        f'<div class="sb"><div class="sv" style="color:#fb923c">{stats["proxy_errors"]}</div><div class="sl">PROXY ERR</div></div>'
        '</div>'
        '<table><thead><tr><th>TIME</th><th>URL</th><th>PROXY</th><th>STATUS</th><th>LAT</th><th>ERROR</th></tr></thead>'
        f'<tbody>{rows}</tbody></table>'
        f'<div style="margin-top:10px;color:#334155;font-size:10px">'
        f'Log: {LOG_FILE} &nbsp;|&nbsp; Proxies: {_proxy_pool_size} &nbsp;|&nbsp; '
        f'Concurrent: {conc_lbl} &nbsp;|&nbsp; Duration: {dur_lbl} &nbsp;|&nbsp; Total: {tot_lbl}'
        '</div></div>'
    )


_last_cfg = {}

def _read_cfg():
    raw_urls = [u.strip() for u in txt_urls.value.strip().splitlines() if u.strip()]
    raw_extra = [u.strip() for u in txt_extra_proxies.value.strip().splitlines() if u.strip()]
    return {
        'urls':       raw_urls if raw_urls else ['https://example.com'],
        'concurrent': int(txt_concurrent.value) if str(txt_concurrent.value).isdigit() else 50,
        'duration':   int(txt_duration.value)   if str(txt_duration.value).isdigit()   else 5,
        'total':      int(txt_total.value)       if str(txt_total.value).isdigit()      else 0,
        'extra':      raw_extra,
    }


out          = widgets.Output()
status_label = widgets.Label('Configure above then press ▶ Start.')

lbl_style = {'description_width': '160px'}
box_layout = widgets.Layout(width='520px')

txt_urls = widgets.Textarea(
    value='https://example.com',
    placeholder='One URL per line',
    description='Target URLs:',
    layout=widgets.Layout(width='520px', height='90px'),
    style=lbl_style
)
txt_concurrent = widgets.Text(
    value='50',
    description='Max Concurrent (0=∞):',
    layout=box_layout, style=lbl_style
)
txt_duration = widgets.Text(
    value='5',
    description='Visit Duration sec (0=instant):',
    layout=box_layout, style=lbl_style
)
txt_total = widgets.Text(
    value='0',
    description='Total Requests (0=∞):',
    layout=box_layout, style=lbl_style
)
txt_extra_proxies = widgets.Textarea(
    value='',
    placeholder='Optional: one proxy per line  e.g. 1.2.3.4:8080\nLeave empty to use free proxies only',
    description='Extra Proxies:',
    layout=widgets.Layout(width='520px', height='70px'),
    style=lbl_style
)

btn_start   = widgets.Button(description='▶ Start',  button_style='success', layout=widgets.Layout(width='110px'))
btn_pause   = widgets.Button(description='⏸ Pause',  button_style='warning', layout=widgets.Layout(width='110px'), disabled=True)
btn_resume  = widgets.Button(description='▶ Resume', button_style='info',    layout=widgets.Layout(width='110px'), disabled=True)
btn_stop    = widgets.Button(description='■ Stop',   button_style='danger',  layout=widgets.Layout(width='110px'), disabled=True)
btn_refresh = widgets.Button(description='↻ Refresh',                         layout=widgets.Layout(width='110px'))

separator = widgets.HTML('<hr style="border:none;border-top:1px solid #333;margin:10px 0">')


def _update_buttons(state):
    btn_start.disabled  = state in ('running', 'paused')
    btn_pause.disabled  = state != 'running'
    btn_resume.disabled = state != 'paused'
    btn_stop.disabled   = state == 'stopped'
    for w in [txt_urls, txt_concurrent, txt_duration, txt_total, txt_extra_proxies]:
        w.disabled = state in ('running', 'paused')


def do_refresh(_=None):
    with out:
        clear_output(wait=True)
        display(HTML(render_dashboard(_last_cfg)))


def _fetch_and_launch(cfg):
    global _blaster_thread, _proxy_pool_size
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    proxy_pool = loop.run_until_complete(_get_proxy_pool(cfg['extra']))
    loop.close()
    _proxy_pool_size = len(proxy_pool)
    status_label.value = f'Running — {_proxy_pool_size} proxies loaded. Click ↻ Refresh to update dashboard.'
    _blaster_thread = threading.Thread(target=_thread_target, args=(proxy_pool, cfg), daemon=True)
    _blaster_thread.start()


def on_start(_):
    global _last_cfg
    if _blaster_thread and _blaster_thread.is_alive():
        status_label.value = 'Already running.'
        return
    cfg = _read_cfg()
    _last_cfg = cfg
    _stop_event.clear()
    _pause_event.clear()
    stats.update({'success': 0, 'failed': 0, 'total_sent': 0, 'proxy_errors': 0,
                  'start_time': time.time(), 'pause_start': 0, 'paused_total': 0})
    recent_log.clear()
    status_label.value = 'Fetching proxies from 5 sources…'
    _update_buttons('running')
    t = threading.Thread(target=_fetch_and_launch, args=(cfg,), daemon=True)
    t.start()
    do_refresh()


def on_pause(_):
    if not _pause_event.is_set():
        _pause_event.set()
        stats['pause_start'] = time.time()
        status_label.value = 'Paused — active requests finish, new ones wait.'
        _update_buttons('paused')
        do_refresh()


def on_resume(_):
    if _pause_event.is_set():
        stats['paused_total'] += time.time() - stats['pause_start']
        _pause_event.clear()
        status_label.value = 'Resumed.'
        _update_buttons('running')
        do_refresh()


def on_stop(_):
    _stop_event.set()
    _pause_event.clear()
    status_label.value = 'Stopping… finishing active requests.'
    _update_buttons('stopped')
    do_refresh()


btn_start.on_click(on_start)
btn_pause.on_click(on_pause)
btn_resume.on_click(on_resume)
btn_stop.on_click(on_stop)
btn_refresh.on_click(do_refresh)

display(
    widgets.HTML('<h3 style="font-family:monospace;color:#38bdf8">⚡ URL Blaster — Config</h3>'),
    txt_urls,
    txt_concurrent,
    txt_duration,
    txt_total,
    txt_extra_proxies,
    separator,
    widgets.HBox([btn_start, btn_pause, btn_resume, btn_stop, btn_refresh]),
    status_label,
    out
)
print('UI ready. Fill in the fields above and click ▶ Start.')
